In [1]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

In [2]:
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
from langchain_groq import ChatGroq

llm=ChatGroq(model="llama-3.1-8b-instant")

/opt/anaconda3/envs/newenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
llm.invoke("hi").content

'How can I assist you today?'

In [6]:
@tool
def calculate_tip(total_bill:int,tip_percent:int)->int:
    """Calculate Tip"""
    return total_bill * tip_percent * 0.01

In [7]:
inputs = {
    "total_bill": 120,
    "tip_percent": 15
}    

tool_map = {
    "calculate_tip": calculate_tip
}

In [8]:
result=tool_map["calculate_tip"].invoke(inputs)

result

18.0

In [9]:
llm_with_tools=llm.bind_tools([calculate_tip])

In [11]:
query = "How much should I tip on $60 at 20%?"

chat_history = [HumanMessage(content=query)]

In [12]:
response=llm_with_tools.invoke(chat_history)

response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'b5vtkgrk6', 'function': {'arguments': '{"tip_percent":20,"total_bill":60}', 'name': 'calculate_tip'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 229, 'total_tokens': 251, 'completion_time': 0.094540497, 'completion_tokens_details': None, 'prompt_time': 0.019163442, 'prompt_tokens_details': None, 'queue_time': 0.092241548, 'total_time': 0.113703939}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4f58-3c3f-7172-94ad-f3737512e63a-0', tool_calls=[{'name': 'calculate_tip', 'args': {'tip_percent': 20, 'total_bill': 60}, 'id': 'b5vtkgrk6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 229, 'output_tokens': 22, 'total_tokens': 251})

In [15]:
tool_calls = response.tool_calls
tool_name = tool_calls[0]["name"]
tool_args = tool_calls[0]["args"]
tool_call_id = tool_calls[0]["id"]

print(tool_name)
print(tool_args)

calculate_tip
{'tip_percent': 20, 'total_bill': 60}


In [16]:
tool_response=tool_map[tool_name].invoke(tool_args)
tool_response

12.0

In [17]:
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_id)

In [18]:
chat_history.extend([response, tool_message])

In [19]:
finalresponse=llm_with_tools.invoke(chat_history)
finalresponse

AIMessage(content='The tip amount is $12.0.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 264, 'total_tokens': 274, 'completion_time': 0.022387005, 'completion_tokens_details': None, 'prompt_time': 0.024258681, 'prompt_tokens_details': None, 'queue_time': 0.156739631, 'total_time': 0.046645686}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4f5b-6064-70f1-833f-8d26e03c11e4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 264, 'output_tokens': 10, 'total_tokens': 274})

In [21]:
class TipAgent:
    def __init__(self,llm):
        self.llm_with_tools=llm.bind_tools([calculate_tip])
        self.tool_map=tool_map
    def run(self,query:str)->str:
        chat_history=[HumanMessage(content=query)]
        response=self.llm_with_tools.invoke(chat_history)

        tool_calls = response.tool_calls
        tool_name = tool_calls[0]["name"]
        tool_args = tool_calls[0]["args"]
        tool_call_id = tool_calls[0]["id"]

        tool_response=self.tool_map[tool_name].invoke(tool_args)

        tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_id)

        chat_history.extend([response,tool_message])

        final_response=llm_with_tools.invoke(chat_history)

        return final_response.content


In [22]:
agent=TipAgent(llm)

agent.run(query)

'The tip would be $12.'